Select this slide when playing slideshow

In [ ]:
#Setup
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
# !pip3 install -U finbourne-sdk finbourne-sdk-utils

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">What are we going to cover?</p>

<p style="font-family: Open Sans; color: #2B6264">- What is a subscription?</p>

<p style="font-family: Open Sans; color: #2B6264">- How can we subscribe to events using the Notifications Application Programming Interface (API) via the LUSID Website?</p>

<p style="font-family: Open Sans; color: #2B6264">- How can we update a subscription to include a filter on the events of interest using the Notifications API via the Python SDK?</p>

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">So what is a subscription?</p>

<p style="font-family: Open Sans; color: #2B6264">- Rather than being triggered directly off the back of an event, notifications (webhooks, emails, text message etc.) are triggered based on a subscription.</p>

<p style="font-family: Open Sans; color: #2B6264">- A subscription is linked to a single type of event, you cannot have a subscription linked to multiple event types.</p>

<p style="font-family: Open Sans; color: #2B6264">- You can create as many subscriptions as you like to a particular event type. You can subscribe every time the event occurs, or filter on event properties to subscribe only in a particular set of circumstances.</p>

<p style="font-family: Open Sans; color: #2B6264">- You can create as many notifications as you like and attach them to a subscription.</p>

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">Updating a subscription</p>

In [ ]:
import pprint
import finbourne.sdk.services.notifications as ln
import finbourne.sdk.services.notifications.models as ln_models

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

# Authenticate to SDK
secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)

event_types_api = api_factory.build(ln.EventTypesApi)
subscriptions_api = api_factory.build(ln.SubscriptionsApi)
display(event_types_api)
display(subscriptions_api)

In [ ]:
try:
    subscriptions_api.delete_subscription(
        scope="FinbourneUniversity",
        code="PortfolioCreated"
    )
except ApiException as api_exception:
    if api_exception.status == 404:
        pass
    else:
        raise
        
subscriptions_api.create_subscription(
    create_subscription=ln.CreateSubscription(
        id=ln.ResourceId(
            scope="FinbourneUniversity",
            code="PortfolioCreated",
        ),
        displayName="Portfolio Created",
        description="A Portfolio has been Created",
        status="Active",
        matchingPattern=ln.MatchingPattern(
            eventType="TransactionPortfolioCreated",
            filter=None
        )
    )
)

In [4]:
response = subscriptions_api.list_subscriptions(filter="id.scope eq 'FinbourneUniversity'")
pprint.pprint(response)

{'href': 'https://fbn-ctools.lusid.com/notification/api/subscriptions/?filter=id.scope%20eq%20%27FinbourneUniversity%27&sortBy=&limit=2000&page=0AcAACEAaWQuc2NvcGUgZXEgJ0ZpbmJvdXJuZVVuaXZlcnNpdHknAAA%3D',
 'links': [{'description': 'A link to the LUSID Insights website showing all '
                           'logs related to this request',
            'href': 'https://fbn-ctools.lusid.com/app/insights/logs/2026042213-e03a63fa2e814c38a86d35a50a535efa',
            'method': 'GET',
            'relation': 'RequestLogs'}],
 'nextPage': None,
 'previousPage': None,
 'values': [{'createdAt': datetime.datetime(2026, 4, 22, 13, 47, 40, 579074, tzinfo=TzInfo(UTC)),
             'description': 'A Portfolio has been Created',
             'displayName': 'Portfolio Created',
             'href': 'https://fbn-ctools.lusid.com/notification/api/subscriptions/FinbourneUniversity/PortfolioCreated',
             'id': {'code': 'PortfolioCreated', 'scope': 'FinbourneUniversity'},
             'matching

In [5]:
event_types = event_types_api.list_event_types().values
pprint.pprint([event_type for event_type in event_types if event_type.id == "TransactionPortfolioCreated"])

[{'application': 'Lusid',
 'bodySchema': None,
 'description': 'An event of this type is fired whenever a new transaction '
                'portfolio is created in LUSID',
 'displayName': 'Transaction Portfolio Created',
 'headerSchema': None,
 'href': 'https://fbn-ctools.lusid.com/notification/api/eventtypes/TransactionPortfolioCreated',
 'id': 'TransactionPortfolioCreated'}]


In [6]:
subscription = [sub for sub in response.values if sub.id.code == "PortfolioCreated"][0]

updated_subscription_response = subscriptions_api.update_subscription(
    scope=subscription.id.scope,
    code=subscription.id.code,
    update_subscription=ln.UpdateSubscription(
        display_name=subscription.display_name,
        description=subscription.description,
        status=subscription.status,
        matching_pattern=ln.MatchingPattern(
            event_type=subscription.matching_pattern.event_type,
            filter="PortfolioDisplayName eq 'GreatPortfolio'"
        )
    )
)

pprint.pprint(updated_subscription_response)


{'createdAt': datetime.datetime(2026, 4, 22, 13, 47, 40, 579074, tzinfo=TzInfo(UTC)),
 'description': 'A Portfolio has been Created',
 'displayName': 'Portfolio Created',
 'href': 'https://fbn-ctools.lusid.com/notification/api/subscriptions/FinbourneUniversity/PortfolioCreated',
 'id': {'code': 'PortfolioCreated', 'scope': 'FinbourneUniversity'},
 'matchingPattern': {'eventType': 'TransactionPortfolioCreated',
                     'filter': "PortfolioDisplayName eq 'GreatPortfolio'"},
 'modifiedAt': datetime.datetime(2026, 4, 22, 13, 47, 54, 669124, tzinfo=TzInfo(UTC)),
 'status': 'Active',
 'useAsAuth': '00u18np6ytgU7Mq9n2p8',
 'userIdCreated': '00u18np6ytgU7Mq9n2p8',
 'userIdModified': '00u18np6ytgU7Mq9n2p8'}


In [7]:
subscriptions_api.delete_subscription(
    scope=updated_subscription_response.id.scope,
    code=updated_subscription_response.id.code
)

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">What have we covered?</p>

<p style="font-family: Open Sans; color: #2B6264">- Subscriptions are required in order to setup notifications in response to events.</p>

<p style="font-family: Open Sans; color: #2B6264">- A subscription is only linked to a single event type.</p>

<p style="font-family: Open Sans; color: #2B6264">- A subscription can subscribe to all events of a particular type. We setup such a subscription using the Notifications API via the LUSID Website.</p>

<p style="font-family: Open Sans; color: #2B6264">- A subscription can also subscribe to a subset of events of a particular type using a filter expression. We updated our subscription with such a filter using the Notifications API via the Python SDK.</p>